# X04 · Model export, operator fallback and DMA cache coherence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/10_compute_deployment/X04_model_export_operator_fallback_and_dma_cache_coh.ipynb)

**Track:** Compute and deployment · **Skill axis (sheet):** Compute and deployment · **Time:** about 55 min

**Skill Hub concepts:** ONNX, TensorRT, TensorFlow Lite; Operator fallback; Direct memory access and cache coherence

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("X04")

## Map: from a training framework to an accelerator

| Format or runtime | Role | Typical target |
|---|---|---|
| ONNX | framework-neutral graph of standard operators (an **opset**) | interchange; ONNX Runtime on CPU or GPU |
| TensorRT | NVIDIA compiler: fuses layers, picks kernels, float16 or int8 | Jetson, data-center GPUs |
| TensorFlow Lite (LiteRT) | flatbuffer for mobile and microcontrollers, int8 | phones, Cortex-M with CMSIS-NN, Edge TPU |

**Operator fallback:** when the accelerator lacks an operator, the runtime cuts the graph into segments and runs the
unsupported ones on the CPU. Each boundary costs a copy (and often a layout or precision conversion), so one exotic
operator in the middle of a network can erase the speedup.

**DMA and caches:** a direct memory access engine moves data without the CPU and without going through the CPU's data
cache. On cores without hardware coherence (Cortex-M7, many application processors' peripherals):

| Direction | Before or after the transfer | Cache operation | Otherwise |
|---|---|---|---|
| CPU writes buffer, DMA sends it (to device) | before starting the DMA | **clean** (write back dirty lines) | device sends stale memory |
| DMA fills buffer, CPU reads it (from device) | before the CPU reads (and avoid touching it during) | **invalidate** | CPU reads stale cached values |

Maintenance works on whole cache lines, so buffers should be line-aligned and line-sized, or neighbors get corrupted.

In [ ]:
import numpy as np
import torch

## Task 1: export to ONNX and run with ONNX Runtime
Use `torch.onnx.export(model, (example,), path, input_names=["input"], output_names=["output"], dynamo=False)`, then
`onnxruntime.InferenceSession(path, providers=["CPUExecutionProvider"])`.

In [ ]:
def export_onnx(model, example, path):
    raise NotImplementedError  # TODO

def run_onnx(path, x):
    """x: numpy float32 array. Return the first output as numpy."""
    raise NotImplementedError  # TODO

lab.check("onnx", export_onnx, run_onnx)

## Task 2: partition a graph for an accelerator
`ops` is the operator list in execution order. Return a list of `(device, [ops])` segments, where device is `"npu"`
for supported operators and `"cpu"` otherwise, merging consecutive operators on the same device.

In [ ]:
def partition(ops, supported):
    raise NotImplementedError  # TODO

lab.check("fallback", partition)

## Task 3: cache maintenance for DMA
`cache_lines(addr, size, line=64)` returns the line-aligned start addresses that cover bytes `[addr, addr + size)`.
`maintenance(direction)` returns `"clean"` for `"to_device"` and `"invalidate"` for `"from_device"`.

In [ ]:
def cache_lines(addr, size, line=64):
    raise NotImplementedError  # TODO

def maintenance(direction):
    raise NotImplementedError  # TODO

lab.check("dma", cache_lines, maintenance)

## Checkpoint: Math
Graph: conv, relu, conv, nms (unsupported), conv, relu, softmax; everything except nms runs on the accelerator.

**M1 (Apply).** Number of segments?

**M2 (Apply).** Cache lines (64 bytes) to maintain for a 1024-byte buffer starting at address 0x20000020?

**M3 (Apply).** Device-to-device transfers (segment boundaries) in that graph?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** An analog-to-digital converter streams samples by DMA into a cached buffer; the CPU sometimes reads
old samples. Missing step?
- a) Clean the buffer's cache lines before reading
- b) Invalidate the buffer's cache lines after the transfer completes, before reading
- c) Disable the DMA interrupt

**K2 (Evaluate).** Your detector runs at 8 ms on the accelerator alone, but 25 ms in the real pipeline because one
custom operator falls back to the CPU. List what that fallback costs and two ways to remove it.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `x04.h`).
3. In `roboprep/checks/x04.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()